# Parsing MBS XML file

In [1]:
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

In [2]:
# Load and parse the XML file
tree = ET.parse('MBS-XML-20260801.xml')
root = tree.getroot()

# View the root tag
print(f"Root tag: {root.tag}")  # Output: Root tag: catalog

Root tag: MBS_XML


In [3]:
len(root)

6046

In [7]:
#for item in root:
#    print(f'New item: {item}')
#    for child in item:
#        print(child.tag)
#        print(child.text)

In [4]:
list_items = []
for i in range(len(root)):
    dict_i = {}
    for child in root[i]:
        dict_i[child.tag] = child.text
    list_items.append(dict_i)

In [5]:
df = pd.DataFrame(list_items)
print(df.shape)

(6046, 40)


In [6]:
df.head(2)

,ItemNum,SubItemNum,ItemStartDate,ItemEndDate,Category,Group,SubGroup,SubHeading,ItemType,FeeType,...,EMSNChangeDate,DescriptionStartDate,Description,QFEStartDate,QFEEndDate,DerivedFeeStartDate,DerivedFee,Benefit75,Benefit85,Anaes
0,3,None,01.12.1989,None,1,A1,None,1,S,N,...,None,01.05.2010,Professional attendance at consulting rooms (o...,None,None,NaN,NaN,NaN,NaN,NaN
1,4,None,01.12.1989,None,1,A1,None,1,S,D,...,None,01.07.2026,Professional attendance by a general practitio...,None,None,01.07.2026,"The fee for item 3, plus $31.50 divided by the...",NaN,NaN,NaN


In [7]:
df.isna().sum()

ItemNum                    0
SubItemNum              6046
ItemStartDate              0
ItemEndDate             6046
Category                   0
Group                      0
SubGroup                1198
SubHeading              4484
ItemType                   0
FeeType                    0
ProviderType            6046
NewItem                    0
ItemChange                 0
AnaesChange                0
DescriptorChange           0
FeeChange                  0
EMSNChange                 0
EMSNCap                    0
BenefitType                0
BenefitStartDate           0
FeeStartDate              84
ScheduleFee               84
Benefit100              5673
BasicUnits              5560
EMSNStartDate           4967
EMSNEndDate             6046
EMSNFixedCapAmount      5966
EMSNMaximumCap          5107
EMSNPercentageCap       5047
EMSNDescription         5997
EMSNChangeDate          6046
DescriptionStartDate       0
Description                0
QFEStartDate            6046
QFEEndDate    

In [14]:
#print(df.dtypes)

In [8]:
numeric_cols = ['ScheduleFee', 'Benefit75', 'Benefit85', 'Benefit100']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
print(df[numeric_cols].dtypes)
print(df[numeric_cols].describe())

ScheduleFee    float64
Benefit75      float64
Benefit85      float64
Benefit100     float64
dtype: object
       ScheduleFee    Benefit75    Benefit85  Benefit100
count  5962.000000  5183.000000  2803.000000  373.000000
mean    666.387487   562.263689   324.160953  127.316220
std     875.181217   681.898923   666.784380  110.669188
min       1.600000     1.200000     1.400000    8.500000
25%     113.062500   108.975000    53.300000   67.400000
50%     331.250000   315.000000   125.250000  104.550000
75%     887.850000   748.750000   322.400000  158.900000
max    9999.950000  7500.000000  9895.450000  883.800000


In [9]:
# Converting ItemStartDate column as date
df['ItemStartDate'] = pd.to_datetime(df['ItemStartDate'], format='%d.%m.%Y')
print(df['ItemStartDate'].dtype)
print(df['ItemStartDate'].min(), '-', df['ItemStartDate'].max())

datetime64[ns]
1984-02-01 00:00:00 - 2026-08-01 00:00:00


In [10]:
df['Category'].value_counts()

Category
3     3409
1      685
6      622
5      540
8      301
4      228
2      139
10      76
7       46
Name: count, dtype: int64

## Category Code → Category Name Mapping
The MBS XML file only provides numeric `Category` codes (1, 2, 3, ... 10) with no text labels. Category names below were verified against official primary sources.
**Categories 1–8** — sourced from the MBS Book (July 2026), published by MBS Online / Department of Health, Disability and Ageing. Table of Contents lists each category by its official chapter title.
- Downloads index: https://www.mbsonline.gov.au/internet/mbsonline/publishing.nsf/Content/downloads
- Local copy used: `PDF_Version_-_MBS_Book_-_July_2026.pdf` (Table of Contents, p.4)

**Category 9** — Not found in the MBS book or in the XML data.

**Category 10 (Dental Services / Child Dental Benefits Schedule)** — sits under different legislation to Categories 1–8 (Health Insurance Act 1973 vs. Dental Benefits Act 2008), which is why it's absent from the MBS Book. 
Verified against:
- Federal Register of Legislation — *Dental Benefits Rules 2014* (as amended), Schedule 1: https://legislation.gov.au/F2022L01625/asmade/2022-12-12/text/original/pdf
- Supporting extract (Dept. of Health): https://www.health.gov.au/sites/default/files/2024-12/dental-benefits-schedule-items-and-rates-for-2025-mbsonline-extract.pdf

In [11]:
# Mapping category codes to category names. See markdown note above for reference info.
category_names = {
    '1': 'Professional Attendances',
    '2': 'Diagnostic Procedures and Investigations',
    '3': 'Therapeutic Procedures',
    '4': 'Oral and Maxillofacial Services',
    '5': 'Diagnostic Imaging Services',
    '6': 'Pathology Services',
    '7': 'Cleft and Craniofacial Services',
    '8': 'Miscellaneous Services',
    '10': 'Dental Services (Child Dental Benefits Schedule)',
}
df['CategoryName'] = df['Category'].map(category_names)

In [12]:
df['FeeType'].value_counts()

FeeType
N    5962
D      84
Name: count, dtype: int64

In [13]:
feetype_D = df[df['FeeType']=='D']
feetype_D = feetype_D.copy()
feetype_D.shape

(84, 41)

In [14]:
feetype_ND = df[df['FeeType']!='D']
feetype_ND = feetype_ND.copy()
feetype_ND.shape

(5962, 41)

## BenefitType → Applicable Benefit Tier(s)
Source: `202202-XML-Field-Descriptions.pdf` (MBS Online, February 2022), `BenefitType` field description — defines the possible values:
A = 75% only, B = 85% only, C = 75% and 85%, D = 75% and 100%, E = 100% only.

In [15]:
feetype_ND.columns

Index(['ItemNum', 'SubItemNum', 'ItemStartDate', 'ItemEndDate', 'Category',
       'Group', 'SubGroup', 'SubHeading', 'ItemType', 'FeeType',
       'ProviderType', 'NewItem', 'ItemChange', 'AnaesChange',
       'DescriptorChange', 'FeeChange', 'EMSNChange', 'EMSNCap', 'BenefitType',
       'BenefitStartDate', 'FeeStartDate', 'ScheduleFee', 'Benefit100',
       'BasicUnits', 'EMSNStartDate', 'EMSNEndDate', 'EMSNFixedCapAmount',
       'EMSNMaximumCap', 'EMSNPercentageCap', 'EMSNDescription',
       'EMSNChangeDate', 'DescriptionStartDate', 'Description', 'QFEStartDate',
       'QFEEndDate', 'DerivedFeeStartDate', 'DerivedFee', 'Benefit75',
       'Benefit85', 'Anaes', 'CategoryName'],
      dtype='object')

In [16]:
id_vars = ['ItemNum', 'ItemStartDate', 'Category', 'CategoryName', 'Group', 'BenefitType','ScheduleFee',
           'Description', 'ProviderType']

In [17]:
df_long = pd.melt(
    feetype_ND, 
    id_vars= id_vars,                 
    value_vars=['Benefit100', 'Benefit75', 'Benefit85'],
    var_name='BenefitTier',                     
    value_name='BenefitAmount'                   
)
df_long.head()

,ItemNum,ItemStartDate,Category,CategoryName,Group,BenefitType,ScheduleFee,Description,ProviderType,BenefitTier,BenefitAmount
0,3,1989-12-01,1,Professional Attendances,A1,E,20.55,Professional attendance at consulting rooms (o...,None,Benefit100,20.55
1,23,1989-12-01,1,Professional Attendances,A1,E,45.05,Professional attendance by a general practitio...,None,Benefit100,45.05
2,36,1989-12-01,1,Professional Attendances,A1,E,87.10,Professional attendance by a general practitio...,None,Benefit100,87.10
3,44,1989-12-01,1,Professional Attendances,A1,E,128.35,Professional attendance by a general practitio...,None,Benefit100,128.35
4,52,1989-12-01,1,Professional Attendances,A2,E,11.00,Professional attendance at consulting rooms of...,None,Benefit100,11.00


In [18]:
df_long.shape

(17886, 11)

In [19]:
df_long = df_long.dropna(subset=['BenefitAmount'])
df_long.shape

(8359, 11)

In [20]:
df_long['BenefitTier'].value_counts()

BenefitTier
Benefit75     5183
Benefit85     2803
Benefit100     373
Name: count, dtype: int64

In [42]:
#df_long[df_long['BenefitTier']=='Benefit75'].head()

In [21]:
df_long['FeeGap'] = df_long['ScheduleFee'] - df_long['BenefitAmount']
df_long['GapPct'] = (df_long['FeeGap'] / df_long['ScheduleFee'] * 100).round(1)

In [22]:
df_long[df_long['BenefitTier'] == 'Benefit100']['FeeGap'].describe()

count    373.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: FeeGap, dtype: float64

In [23]:
df_long = df_long.drop(columns=['ProviderType'])

## Group Code → Group Name Mapping

**Groups A1–T (91 non-dental codes)** — sourced from the MBS Book (July 2026),
Table of Contents (mbsonline.gov.au). Local copy: `PDF_Version_-_MBS_Book_-_July_2026.pdf`

**Groups U0–U9 (8 dental codes)** — sourced from the Federal Register of Legislation, Dental Benefits Rules 2014 (consolidated), Schedule 1, Table of Contents. (https://www.legislation.gov.au/F2026L00110)

In [24]:
#!pip install pypdf

In [25]:
from pypdf import PdfReader

reader = PdfReader("PDF Version - MBS Book - July 2026.pdf")
print(len(reader.pages))          # total page count

1760


In [56]:
#print(reader.pages[3].extract_text())   # raw text of page 4 (0-indexed)

In [26]:
pattern = r'Group\s+([A-Z]?\d+[A-Z]?)\.\s*(.+?)\s*\.{2,}'

In [27]:
actual_groups = set(feetype_ND['Group'].dropna().unique())
dental = {'U0','U1','U2','U3','U4','U5','U6','U7','U8','U9'}
actual_non_dental = actual_groups - dental
print(len(actual_non_dental))

91


In [28]:
#!pip install regex

In [29]:
import regex as re

In [30]:
toc_text = ''
for i in range(3, 45):
    toc_text += reader.pages[i].extract_text() + '\n'

matches = re.findall(pattern, toc_text, flags=re.DOTALL)
print(len(matches))

91


In [31]:
#matched_codes = set(code for code, name in matches)
#missing = actual_non_dental - matched_codes
#print(missing)

In [32]:
#matches = re.findall(pattern, reader.pages[3].extract_text())
# matches[0] is the first full match, e.g. ('A1', 'General Practitioner Attendances...')
print(matches[0][0])   # 'A1'      <- like $1 in R
print(matches[0][1])   # 'General Practitioner Attendances...'   <- like $2

A1
General Practitioner Attendances To Which No Other Item Applies


In [33]:
grp_dict = {}
#for i in range(len(matches)):
#    print(matches[i])
#    lst = matches[i]
#    grp_dict[lst[0]] = lst[1]

for code, name in matches:
    grp_dict[code] = name

In [34]:
grp_dict

{'A1': 'General Practitioner Attendances To Which No Other Item Applies',
 'A2': 'Other Non-Referred Attendances To Which No Other Item Applies',
 'A3': 'Specialist Attendances To Which No Other Item Applies',
 'A4': 'Consultant Physician Attendances To Which No Other Item Applies',
 'A5': 'Prolonged Attendances To Which No Other Item Applies',
 'A6': 'Group Therapy',
 'A7': 'Acupuncture and Non-Specialist Practitioner Items',
 'A8': 'Consultant Psychiatrist Attendances To Which No Other Item Applies',
 'A9': 'Contact Lenses - Attendances',
 'A10': 'Optometrical Services',
 'A11': 'Urgent Attendance After Hours',
 'A12': 'Consultant Occupational Physician Attendances To Which No Other Item Applies',
 'A13': 'Public Health Physician Attendances To Which No Other Item Applies',
 'A14': 'Health Assessments',
 'A15': 'GP chronic condition management plans, multidisciplinary care plans and case conferences',
 'A17': 'Domiciliary And Residential Management Reviews',
 'A20': 'GP Mental Health

In [35]:
print(len(grp_dict))
#print(grp_dict)

91


In [36]:
dental_groups = {
    'U0': 'Diagnostic Services',
    'U1': 'Preventive Services',
    'U2': 'Periodontic Services',
    'U3': 'Oral Surgery',
    'U4': 'Endodontic Services',
    'U5': 'Restorative Services',
    'U7': 'Prosthodontics',
    'U9': 'General Services'
}

In [37]:
grp_dict.update(dental_groups)

In [38]:
len(grp_dict)

99

In [39]:
df_long['GroupName'] = df_long['Group'].map(grp_dict)
print(df_long['GroupName'].isna().sum())

0


In [40]:
df_long.to_csv('mbs_gap_analysis.csv', index=False)

check = pd.read_csv('mbs_gap_analysis.csv')
print(check.shape)
print(check.dtypes)

(8359, 13)
ItemNum            int64
ItemStartDate     object
Category           int64
CategoryName      object
Group             object
BenefitType       object
ScheduleFee      float64
Description       object
BenefitTier       object
BenefitAmount    float64
FeeGap           float64
GapPct           float64
GroupName         object
dtype: object
